In [1]:
import os
import re
import glob
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import scipy.io as sio
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from spikingjelly.activation_based import neuron, functional, surrogate, layer

ImportError: DLL load failed while importing _C: No se puede encontrar el módulo especificado.

In [ ]:
# ============================================================
# CONFIGURACION
# ============================================================
# Cambia esta ruta si hace falta
DATA_DIR = r"C:\Users\Usuario\OneDrive\Documentos\UNI\TFG\tfg-snn-monica\data\ExtractedFeatures_1s"

# Si tus etiquetas no salen de los ficheros, puedes ponerlas aqui manualmente.
# En SEED clasico suele haber 15 peliculas y 3 clases: 0=negativa, 1=neutra, 2=positiva
# Ajusta este vector si tu version usa otro orden.
DEFAULT_SEED_LABELS = np.array([
    2, 1, 0, 0, 1,
    2, 0, 1, 2, 1,
    2, 0, 1, 2, 0
], dtype=np.int64)

SEED = 42
BATCH_SIZE = 128
EPOCHS = 40
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
T = 8  # timesteps de la SNN
VAL_SIZE = 0.15
TEST_SIZE = 0.15
HIDDEN1 = 256
HIDDEN2 = 128
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# ============================================================
# UTILIDADES
# ============================================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def list_mat_files(data_dir: str) -> List[str]:
    files = sorted(glob.glob(os.path.join(data_dir, '*.mat')))
    if not files:
        raise FileNotFoundError(f'No se encontraron archivos .mat en: {data_dir}')
    return files


def is_valid_feature_array(arr: np.ndarray) -> bool:
    if not isinstance(arr, np.ndarray):
        return False
    if arr.dtype == object:
        return False
    if arr.ndim < 2:
        return False
    if min(arr.shape) < 2:
        return False
    return np.issubdtype(arr.dtype, np.number)


def infer_trial_index_from_key(key: str) -> Optional[int]:
    # Busca un numero al final del nombre, por ejemplo de_LDS1 -> 1
    m = re.search(r'(\d+)$', key)
    if m:
        return int(m.group(1)) - 1
    return None


def flatten_feature_matrix(arr: np.ndarray) -> np.ndarray:
    # Convierte cada ventana temporal en un vector 1D
    # Casos tipicos:
    #   (n_segments, n_features)
    #   (n_features, n_segments)
    #   (n_channels, n_bands, n_segments)
    arr = np.asarray(arr)

    if arr.ndim == 2:
        # Queremos que la primera dimension sea muestras
        if arr.shape[0] < arr.shape[1]:
            # normal en SEED puede ser (features, segments), lo transponemos
            arr = arr.T
        return arr.reshape(arr.shape[0], -1).astype(np.float32)

    # Para 3D o mas, intentamos poner segmentos delante
    # Heuristica: la ultima dimension suele ser n_segments
    last_dim = arr.shape[-1]
    arr = np.moveaxis(arr, -1, 0)
    return arr.reshape(arr.shape[0], -1).astype(np.float32)


@dataclass
class LoadedData:
    X: np.ndarray
    y: np.ndarray
    meta: List[Dict]
    feature_dim: int

In [ ]:
# ============================================================
# CARGADOR ROBUSTO PARA SEED / EXTRACTED FEATURES
# ============================================================
def load_seed_extracted_features(
    data_dir: str,
    label_vector: Optional[np.ndarray] = None,
    verbose: bool = True
) -> LoadedData:
    """
    Carga de forma robusta los .mat de SEED con features extraidas.

    Intenta detectar claves de features automaticamente. Si no encuentra etiquetas
    dentro del fichero, usa label_vector por indice de trial.
    """
    files = list_mat_files(data_dir)
    if label_vector is None:
        label_vector = DEFAULT_SEED_LABELS

    all_X = []
    all_y = []
    all_meta = []

    for file_path in files:
        mat = sio.loadmat(file_path)
        basename = os.path.basename(file_path)

        keys = [k for k in mat.keys() if not k.startswith('__')]
        if verbose:
            print(f'\nArchivo: {basename}')
            print(f'Claves encontradas: {keys}')

        used_any = False

        for key in keys:
            arr = mat[key]
            if not is_valid_feature_array(arr):
                continue

            trial_idx = infer_trial_index_from_key(key)
            if trial_idx is None or trial_idx >= len(label_vector):
                # Si no hay indice claro, lo saltamos para evitar mezclar cosas raras
                continue

            X_trial = flatten_feature_matrix(arr)
            y_trial = np.full(shape=(X_trial.shape[0],), fill_value=int(label_vector[trial_idx]), dtype=np.int64)

            all_X.append(X_trial)
            all_y.append(y_trial)
            all_meta.extend([
                {
                    'file': basename,
                    'key': key,
                    'trial_idx': int(trial_idx),
                    'label': int(label_vector[trial_idx])
                }
                for _ in range(X_trial.shape[0])
            ])
            used_any = True

            if verbose:
                print(f'  Usando clave {key}: forma original {arr.shape} -> muestras {X_trial.shape}')

        if not used_any and verbose:
            print(f'  Aviso: no se pudo usar ninguna clave valida en {basename}')

    if not all_X:
        raise RuntimeError(
            'No se pudieron cargar features validas. '\
            'Revisa los nombres de las claves dentro de los .mat y el vector de etiquetas.'
        )

    X = np.concatenate(all_X, axis=0)
    y = np.concatenate(all_y, axis=0)

    if verbose:
        print('\nResumen de carga:')
        print(f'  X shape = {X.shape}')
        print(f'  y shape = {y.shape}')
        print(f'  Clases y conteos = {dict(zip(*np.unique(y, return_counts=True)))}')

    return LoadedData(X=X, y=y, meta=all_meta, feature_dim=X.shape[1])


In [ ]:
# ============================================================
# DATASET
# ============================================================
class EEGFeatureDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]


# ============================================================
# CODIFICACION A SPIKES
# ============================================================
def rate_encode_features(x: torch.Tensor, T: int) -> torch.Tensor:
    """
    x: [batch, features] ya normalizado a [0,1]
    devuelve: [T, batch, features]
    """
    x = x.clamp(0.0, 1.0)
    # Bernoulli por cada timestep
    return (torch.rand((T, *x.shape), device=x.device) < x.unsqueeze(0)).float()


# ============================================================
# MODELO SNN
# ============================================================
class SEEDSNN(nn.Module):
    def __init__(self, input_dim: int, hidden1: int = 256, hidden2: int = 128, num_classes: int = 3):
        super().__init__()
        self.fc1 = layer.Linear(input_dim, hidden1)
        self.lif1 = neuron.LIFNode(tau=2.0, surrogate_function=surrogate.ATan())

        self.fc2 = layer.Linear(hidden1, hidden2)
        self.lif2 = neuron.LIFNode(tau=2.0, surrogate_function=surrogate.ATan())

        self.fc3 = layer.Linear(hidden2, num_classes)
        self.lif3 = neuron.LIFNode(tau=2.0, surrogate_function=surrogate.ATan())

    def forward(self, x_seq: torch.Tensor) -> torch.Tensor:
        """
        x_seq: [T, batch, input_dim]
        devuelve logits medios: [batch, num_classes]
        """
        T_steps = x_seq.shape[0]
        out_spikes = []

        for t in range(T_steps):
            x = x_seq[t]
            x = self.fc1(x)
            x = self.lif1(x)
            x = self.fc2(x)
            x = self.lif2(x)
            x = self.fc3(x)
            x = self.lif3(x)
            out_spikes.append(x)

        out_spikes = torch.stack(out_spikes, dim=0)      # [T, B, C]
        logits = out_spikes.mean(dim=0)                  # [B, C]
        return logits

In [ ]:
# ============================================================
# ENTRENAMIENTO Y EVALUACION
# ============================================================
def prepare_data(data: LoadedData):
    X_train, X_tmp, y_train, y_tmp = train_test_split(
        data.X, data.y,
        test_size=VAL_SIZE + TEST_SIZE,
        random_state=SEED,
        stratify=data.y
    )

    relative_test_size = TEST_SIZE / (VAL_SIZE + TEST_SIZE)
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp,
        test_size=relative_test_size,
        random_state=SEED,
        stratify=y_tmp
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    # Pasamos de normalizacion z-score a [0,1] para codificacion rate
    min_v = X_train.min(axis=0, keepdims=True)
    max_v = X_train.max(axis=0, keepdims=True)
    span = np.where((max_v - min_v) < 1e-8, 1.0, (max_v - min_v))

    X_train = (X_train - min_v) / span
    X_val = (X_val - min_v) / span
    X_test = (X_test - min_v) / span

    X_train = np.clip(X_train, 0.0, 1.0)
    X_val = np.clip(X_val, 0.0, 1.0)
    X_test = np.clip(X_test, 0.0, 1.0)

    return X_train, X_val, X_test, y_train, y_val, y_test


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, T: int):
    model.eval()
    all_preds = []
    all_true = []
    loss_fn = nn.CrossEntropyLoss()
    losses = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        x_seq = rate_encode_features(x, T=T)
        logits = model(x_seq)
        loss = loss_fn(logits, y)
        losses.append(loss.item())

        preds = torch.argmax(logits, dim=1)
        all_preds.append(preds.cpu().numpy())
        all_true.append(y.cpu().numpy())

        functional.reset_net(model)

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_preds)

    return {
        'loss': float(np.mean(losses)),
        'acc': accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'y_true': y_true,
        'y_pred': y_pred,
    }


def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, T: int):
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()

    best_state = None
    best_val_acc = -1.0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []

        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()
            x_seq = rate_encode_features(x, T=T)
            logits = model(x_seq)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            functional.reset_net(model)

        val_metrics = evaluate(model, val_loader, T=T)

        if val_metrics['acc'] > best_val_acc:
            best_val_acc = val_metrics['acc']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f'Epoch {epoch:03d} | '
            f'train_loss={np.mean(train_losses):.4f} | '
            f'val_loss={val_metrics["loss"]:.4f} | '
            f'val_acc={val_metrics["acc"]:.4f} | '
            f'val_f1={val_metrics["f1_macro"]:.4f}'
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

In [ ]:
# ============================================================
# MAIN
# ============================================================
def main():
    set_seed(SEED)
    print(f'Usando dispositivo: {DEVICE}')

    data = load_seed_extracted_features(DATA_DIR, label_vector=DEFAULT_SEED_LABELS, verbose=True)
    X_train, X_val, X_test, y_train, y_val, y_test = prepare_data(data)

    train_ds = EEGFeatureDataset(X_train, y_train)
    val_ds = EEGFeatureDataset(X_val, y_val)
    test_ds = EEGFeatureDataset(X_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    model = SEEDSNN(
        input_dim=data.feature_dim,
        hidden1=HIDDEN1,
        hidden2=HIDDEN2,
        num_classes=len(np.unique(data.y))
    ).to(DEVICE)

    print(f'\nEntrenando SNN con input_dim={data.feature_dim}, T={T}')
    model = train_model(model, train_loader, val_loader, T=T)

    print('\nEvaluacion final en test:')
    test_metrics = evaluate(model, test_loader, T=T)
    print(f"Test loss: {test_metrics['loss']:.4f}")
    print(f"Test acc : {test_metrics['acc']:.4f}")
    print(f"Test F1  : {test_metrics['f1_macro']:.4f}")

    print('\nClassification report:')
    print(classification_report(test_metrics['y_true'], test_metrics['y_pred'], digits=4))

    print('Confusion matrix:')
    print(confusion_matrix(test_metrics['y_true'], test_metrics['y_pred']))

    os.makedirs('checkpoints', exist_ok=True)
    ckpt_path = os.path.join('checkpoints', 'seed_snn_spikingjelly.pth')
    torch.save({
        'model_state_dict': model.state_dict(),
        'input_dim': data.feature_dim,
        'T': T,
        'hidden1': HIDDEN1,
        'hidden2': HIDDEN2,
    }, ckpt_path)
    print(f'\nModelo guardado en: {ckpt_path}')


if __name__ == '__main__':
    main()